# WeightedKgBlend — Novel Drug Repurposing Candidates

Identifies novel repurposing candidates from slice_0 test predictions.

**Strategy:** Use Path-Gated predictions (RotatE rank + ProbCBR mechanistic path) to surface
drug-disease pairs that are:
1. Highly ranked by RotatE (embedding confidence)
2. Supported by a mechanistic biological path (ProbCBR)
3. NOT the held-out known indication (novel predictions)

**Evidence tiers:**
- **Tier 1** — RotatE rank ≤ 5 + path exists  
- **Tier 2** — RotatE rank ≤ 10 + path exists  
- **Tier 3** — RotatE rank ≤ 20 + path exists

In [ ]:
import pandas as pd
import numpy as np
import urllib.request
import json
import time
from pathlib import Path
from collections import defaultdict

RESULTS = Path('/Users/meghamala/projects/WeightedKgBlend/results/predictions')
SLICE   = 'slice_0'

# Optimised weights from 05_path_gated_reranking.ipynb
ALPHA = 0.0008   # RotatE reciprocal-rank weight
BETA  = 0.8845   # ProbCBR path-score weight

# ── Load path lookup (RotatE candidates scored by ProbCBR) ────────────────
df_test  = pd.read_csv(RESULTS / 'PathGated' / SLICE / 'path_lookup_rotate_test_slice0.tsv',  sep='\t')
df_valid = pd.read_csv(RESULTS / 'PathGated' / SLICE / 'path_lookup_rotate_valid_slice0.tsv', sep='\t')

print(f'Test  path lookup : {len(df_test):,} rows')
print(f'Valid path lookup : {len(df_valid):,} rows')
print(f'Columns: {df_test.columns.tolist()}')

In [ ]:
# ── Build novel candidate table ───────────────────────────────────────────
# Combine test + valid (both are novel predictions from different drugs)
df_all = pd.concat([df_test, df_valid], ignore_index=True)

# Keep only non-expected (novel) drug-disease pairs
novel = df_all[df_all['is_expected'] == False].copy()

# Aggregate: sum path scores, keep best path, count paths per pair
candidates = (
    novel.groupby(['drug', 'disease'])
    .agg(
        rotate_rank      = ('rotate_rank', 'first'),
        total_path_score = ('path_score', 'sum'),
        n_paths          = ('path', 'count'),
        best_path        = ('path', 'first'),   # top-scored path (already sorted by path_rank)
        split            = ('split', 'first'),
    )
    .reset_index()
)

# Path-Gated combined score (same formula as evaluation)
candidates['pg_score'] = (
    ALPHA * (1.0 / candidates['rotate_rank']) +
    BETA  * candidates['total_path_score']
)

# Tier assignment
def assign_tier(rank):
    if rank <= 5:  return 'Tier 1'
    if rank <= 10: return 'Tier 2'
    if rank <= 20: return 'Tier 3'
    return 'Tier 4'  # rank 21-50 (lower confidence)

candidates['tier'] = candidates['rotate_rank'].apply(assign_tier)
candidates = candidates.sort_values(['tier', 'pg_score'], ascending=[True, False]).reset_index(drop=True)

print(f'Total novel candidates with mechanistic paths: {len(candidates):,}')
print(f'Unique drugs  : {candidates["drug"].nunique():,}')
print(f'Unique diseases: {candidates["disease"].nunique():,}')
print()
print('Tier breakdown:')
print(candidates['tier'].value_counts().sort_index().to_string())

In [ ]:
# ── Fetch human-readable names via OLS4 / NCBI APIs ──────────────────────

_name_cache = {}

def _ols_lookup(iri, timeout=8):
    url = f'https://www.ebi.ac.uk/ols4/api/terms?iri={urllib.parse.quote(iri, safe=":/")}&exact=true'
    try:
        with urllib.request.urlopen(url, timeout=timeout) as r:
            data = json.loads(r.read())
        terms = data.get('_embedded', {}).get('terms', [])
        if terms:
            return terms[0].get('label', None)
    except Exception:
        pass
    return None

def _mesh_lookup(mesh_id, timeout=8):
    """MeSH term lookup via NLM API."""
    url = f'https://id.nlm.nih.gov/mesh/lookup/descriptor?label={mesh_id}&match=exact&limit=1'
    try:
        with urllib.request.urlopen(url, timeout=timeout) as r:
            data = json.loads(r.read())
        if data:
            return data[0].get('label', None)
    except Exception:
        pass
    return None

import urllib.parse

def get_name(entity_id):
    if entity_id in _name_cache:
        return _name_cache[entity_id]
    
    prefix, local = entity_id.split(':', 1)
    name = None
    
    if prefix == 'CHEBI':
        name = _ols_lookup(f'http://purl.obolibrary.org/obo/CHEBI_{local}')
    elif prefix == 'DOID':
        name = _ols_lookup(f'http://purl.obolibrary.org/obo/DOID_{local}')
    elif prefix == 'MONDO':
        name = _ols_lookup(f'http://purl.obolibrary.org/obo/MONDO_{local}')
    elif prefix == 'MESH':
        # Try OLS first (some MESH terms are there)
        name = _ols_lookup(f'http://id.nlm.nih.gov/mesh/{local}')
        if not name:
            # Fall back to NLM lookup
            url = f'https://id.nlm.nih.gov/mesh/{local}.json'
            try:
                with urllib.request.urlopen(url, timeout=8) as r:
                    data = json.loads(r.read())
                name = data.get('label', {}).get('@value', None)
            except Exception:
                pass
    elif prefix == 'OMIM':
        name = f'OMIM:{local}'  # OMIM requires API key
    elif prefix in ('IKEY', 'UNII'):
        name = entity_id  # No easy free lookup
    elif prefix == 'WD':
        # Wikidata
        url = f'https://www.wikidata.org/wiki/Special:EntityData/{local}.json'
        try:
            with urllib.request.urlopen(url, timeout=8) as r:
                data = json.loads(r.read())
            ent = data.get('entities', {}).get(local, {})
            labels = ent.get('labels', {})
            name = labels.get('en', {}).get('value', None)
        except Exception:
            pass
    
    result = name if name else entity_id
    _name_cache[entity_id] = result
    time.sleep(0.05)  # polite rate limit
    return result


# Collect all unique entities we need to name
tier1_2 = candidates[candidates['tier'].isin(['Tier 1', 'Tier 2'])]
unique_entities = set(tier1_2['drug'].unique()) | set(tier1_2['disease'].unique())

print(f'Fetching names for {len(unique_entities)} unique entities (Tier 1+2)...')
for i, eid in enumerate(sorted(unique_entities)):
    get_name(eid)
    if (i+1) % 20 == 0:
        print(f'  {i+1}/{len(unique_entities)} done...')
print(f'Done. {len(_name_cache)} names fetched.')

In [ ]:
# ── Display top candidates ────────────────────────────────────────────────

def format_path(path_str):
    """Shorten path by replacing entity IDs with names where cached."""
    if not isinstance(path_str, str):
        return ''
    parts = path_str.split(' --')
    out = []
    for part in parts:
        part = part.strip()
        if part.startswith('['):
            out.append(part)
        else:
            name = _name_cache.get(part, part)
            out.append(name)
    return ' → '.join(out)

# Build display table for Tier 1 + 2
display_rows = []
for _, row in tier1_2.iterrows():
    drug_name = _name_cache.get(row['drug'], row['drug'])
    dis_name  = _name_cache.get(row['disease'], row['disease'])
    display_rows.append({
        'Tier'             : row['tier'],
        'Drug'             : drug_name,
        'Drug ID'          : row['drug'],
        'Disease'          : dis_name,
        'Disease ID'       : row['disease'],
        'RotatE rank'      : int(row['rotate_rank']),
        'Path score'       : round(row['total_path_score'], 5),
        'N paths'          : int(row['n_paths']),
        'Best mechanistic path' : row['best_path'],
    })

display_df = pd.DataFrame(display_rows)

print('=' * 80)
print('TOP NOVEL DRUG REPURPOSING CANDIDATES (slice_0, Tier 1 & 2)')
print('=' * 80)
print(f'Tier 1 (RotatE rank ≤ 5): {(display_df["Tier"]=="Tier 1").sum()} pairs')
print(f'Tier 2 (RotatE rank ≤ 10): {(display_df["Tier"]=="Tier 2").sum()} pairs')
print()

# Print Tier 1
t1 = display_df[display_df['Tier'] == 'Tier 1'].copy()
print('── TIER 1 (highest confidence: rank ≤ 5 + mechanistic path) ──')
for _, r in t1.iterrows():
    print(f'  Drug    : {r["Drug"]} ({r["Drug ID"]})')
    print(f'  Disease : {r["Disease"]} ({r["Disease ID"]})')
    print(f'  Rank    : {r["RotatE rank"]}  |  Path score: {r["Path score"]:.5f}  |  N paths: {r["N paths"]}')
    print(f'  Path    : {r["Best mechanistic path"]}')
    print()

In [ ]:
# ── Tier 2 summary table ──────────────────────────────────────────────────
t2 = display_df[display_df['Tier'] == 'Tier 2'].copy()
print('── TIER 2 (rank ≤ 10 + mechanistic path) ──')
print(t2[['Drug', 'Disease', 'RotatE rank', 'Path score', 'N paths']].to_string(index=False))

In [ ]:
# ── Save candidates to TSV ────────────────────────────────────────────────
OUT = Path('/Users/meghamala/projects/WeightedKgBlend/results')

# Save Tier 1+2 with names
display_df.to_csv(OUT / 'repurposing_candidates_tier1_2.tsv', sep='\t', index=False)

# Save all tiers (Tier 1-4) with IDs only (for downstream analysis)
candidates_out = candidates.copy()
candidates_out['drug_name']    = candidates_out['drug'].map(_name_cache)
candidates_out['disease_name'] = candidates_out['disease'].map(_name_cache)
candidates_out.to_csv(OUT / 'repurposing_candidates_all.tsv', sep='\t', index=False)

print(f'Saved {len(display_df)} Tier 1+2 candidates to results/repurposing_candidates_tier1_2.tsv')
print(f'Saved {len(candidates_out)} all-tier candidates to results/repurposing_candidates_all.tsv')

In [ ]:
# ── Drug-centric view: top disease predictions per drug (Tier 1 only) ─────
print('Drug-centric view — Tier 1 predictions:')
print('(For each drug: strongest novel mechanistic repurposing indication)')
print()

for drug_id, group in t1.groupby('Drug ID'):
    drug_name = _name_cache.get(drug_id, drug_id)
    print(f'Drug: {drug_name} ({drug_id})')
    for _, r in group.sort_values('Path score', ascending=False).iterrows():
        print(f'  → {r["Disease"]} (rank {r["RotatE rank"]}, path_score={r["Path score"]:.5f}, {r["N paths"]} paths)')
        print(f'     {r["Best mechanistic path"]}')
    print()